In [2]:
import pandas as pd
import sqlite3

# 1. Load dataset
try:
    df = pd.read_csv("upi_data.csv")
    print(f"Dataset successfully loaded with {len(df)} records.\n")
except Exception as e:
    print(f"Error reading CSV file: {e}")
    exit()

# 2. Basic Data Cleaning
# Clean column names (strip whitespace)
df.columns = df.columns.str.strip()

# Filter for successful transactions if a Status column exists
status_col = [c for c in df.columns if 'status' in c.lower()]
if status_col:
    df = df[df[status_col[0]].astype(str).str.upper() == 'SUCCESS']

# Drop rows with missing values in key columns
amount_col = [c for c in df.columns if 'amount' in c.lower() or 'value' in c.lower()][0]
date_col = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()][0]
category_col = [c for c in df.columns if 'cat' in c.lower() or 'type' in c.lower() or 'merchant' in c.lower()]

df = df.dropna(subset=[amount_col, date_col])

# Rename columns to standard names for SQLite queries
df = df.rename(columns={amount_col: 'Amount', date_col: 'Date'})
if category_col:
    df = df.rename(columns={category_col[0]: 'Category'})
else:
    df['Category'] = 'General Retail'

# Ensure Date is formatted as string for SQLite
df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')

# 3. Create In-Memory SQLite Database
conn = sqlite3.connect(":memory:")
df.to_sql("upi_transactions", conn, index=False, if_exists="replace")

# 4. Query 1: Monthly Growth & Average Ticket Size
print("=" * 50)
print("QUERY 1: Monthly Transaction Volume & Ticket Size")
print("=" * 50)
query_monthly = """
SELECT
    strftime('%Y-%m', Date) AS Month,
    COUNT(*) AS Total_Transactions,
    ROUND(SUM(Amount), 2) AS Total_Spend_INR,
    ROUND(AVG(Amount), 2) AS Avg_Ticket_Size
FROM upi_transactions
GROUP BY Month
ORDER BY Month ASC;
"""
monthly_df = pd.read_sql_query(query_monthly, conn)
print(monthly_df.head(12))

# 5. Query 2: Category Breakdown & Spend Share
print("\n" + "=" * 50)
print("QUERY 2: Spending Share by Merchant Category")
print("=" * 50)
query_category = """
SELECT
    Category,
    COUNT(*) AS Tx_Count,
    ROUND(SUM(Amount), 2) AS Total_Spend_INR,
    ROUND((SUM(Amount) * 100.0 / (SELECT SUM(Amount) FROM upi_transactions)), 2) AS Spend_Share_Pct
FROM upi_transactions
GROUP BY Category
ORDER BY Total_Spend_INR DESC;
"""
category_df = pd.read_sql_query(query_category, conn)
print(category_df)

conn.close()

Dataset successfully loaded with 1000 records.

QUERY 1: Monthly Transaction Volume & Ticket Size
     Month  Total_Transactions  Total_Spend_INR  Avg_Ticket_Size
0  2024-06                 445       2255398.51          5068.31
1  2024-07                  57        284159.18          4985.25

QUERY 2: Spending Share by Merchant Category
         Category  Tx_Count  Total_Spend_INR  Spend_Share_Pct
0  General Retail       502       2539557.69            100.0
